# Settings

In [ ]:
parameters_file_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/params.json"
convert_complex = False

#Simulation settings
using_simulation_data = False
rf_matrix_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/filtered/rf0001.npy"

#Real data settings
using_matlab_data = False       #! high RAM usage for big files -> convert mat file to npy for memory map usage and set using_simulation_data to False
interpolate_data = True

#Batch processing settings
batch_processing = True
batch_size = 1                 #ignored if batch_processing is False
frames_to_process = 160
MAX_WORKERS = 10                 #max is 12 for the current computer (10 max recommended)

#Beamforming settings
run_nsi_apodization = True
nsi_dc_offset = 0.01
other_apodization = "rectangle".lower()

#Saving parameters
save_image = True
bf_data_path = "./data_bf/"
folder_name = "test_classes"
beamformed_apodized_matrix_name = "001_test"
angles_debbuging = False          #if true saves the angles used for the beamforming (z,x, angles, frames)


# Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json
import timeit
import os
import math
import concurrent.futures
from numba import njit
from scipy.signal import envelope, firwin, hilbert
from scipy import fft
from scipy.io import loadmat

import modules.beamforming as bf

# Import parameters

In [3]:
def load_parameters(json_file_path):
    with open(json_file_path, 'r') as f:
        params = json.load(f)
    return params


In [4]:
def add_extra_params(params):
    
    params["rf"]["upsampled_samples"] = params["rf"]["samples"] + (params["rf"]["interpolation_factor"] - 1) * (params["rf"]["samples"] - 1)
    params["rf"]["upsampled_sampling_rate"] = params["rf"]["sampling_rate"] * params["rf"]["interpolation_factor"] 

    return params

In [5]:
params = load_parameters(parameters_file_path)
params = add_extra_params(params)

pos = np.linspace(start=0, stop=params["array"]["length"], num=params["array"]["length"], dtype=int) #array position sampling
time = np.linspace(start=0, stop=1 / params["rf"]["sampling_rate"] * params["rf"]["samples"], num=params["rf"]["samples"]) #time sampling
C = params["rf"]["C"] #speed of sound

print(f"Max acquisition depth : {params["array"]["acq_depth"][1]} m") #v = d / t => d = v * t / 2 (round trip) with t = sample_step * samples

Max acquisition depth : 0.012 m


# Create workspace

In [6]:
if not os.path.exists(f"{bf_data_path}{folder_name}"):
    os.makedirs(f"{bf_data_path}{folder_name}")
    print(f"Folder {bf_data_path}{folder_name} doesn't exist. Setting up the folder.")

os.makedirs(f"{bf_data_path}{folder_name}/beamformed_final", exist_ok=True)
os.makedirs(f"{bf_data_path}{folder_name}/beamformed_apodized", exist_ok=True)
os.makedirs(f"{bf_data_path}{folder_name}/images", exist_ok=True)

# Run beamforming

In [ ]:
beamforming = bf.Beamforming(
    nsi_dc_offset=nsi_dc_offset,
    params=params, 
    debug=False,
    angles_debug=False,
    MAX_WORKERS=MAX_WORKERS, 
    frames_to_process=frames_to_process, 
    batch_size=batch_size, 
    rf_matrix_path=rf_matrix_path, 
    using_simulation_data=using_simulation_data, 
    using_matlab_data=using_matlab_data, 
    convert_complex=convert_complex,
    interpolate_data=interpolate_data,
    run_nsi_apodization=run_nsi_apodization,
    other_apodization=other_apodization,
    bf_data_path=bf_data_path,
    folder_name=folder_name,
    beamformed_apodized_matrix_name=beamformed_apodized_matrix_name,
    )

beamforming.run_beamforming()

Frame iterations: 160
Batch processing with 1 frames
Samples for acquisition depth: 200
Samples for array length: 256
Beamformed data of shape (200, 256, 160) saved at: ./data_bf/test_classes/beamformed_apodized/001_test_small_zm.npy
Beamformed data of shape (200, 256, 160) saved at: ./data_bf/test_classes/beamformed_apodized/001_test_small_dc1.npy
Beamformed data of shape (200, 256, 160) saved at: ./data_bf/test_classes/beamformed_apodized/001_test_small_dc2.npy

Starting process pool with 10 workers.
Total frames to process: 160
Batch size: 1
Total batches: 160
Processing batch 1 [1/160] (Process ID: 190131)
  • Processing from frame 0 to frame 1
  • Processing 1 frame(s)
  • RF matrix: Imported RF matrix with shape:  (1535, 128, 16, 1)
  • Running NSI apodizations
Processing batch 2 [2/160] (Process ID: 190132)
  • Processing from frame 1 to frame 2
  • Processing 1 frame(s)
  • RF matrix: Imported RF matrix with shape:  (1535, 128, 16, 1)
  • Running NSI apodizations
Processing bat

# Apply NSI 

In [ ]:
# def get_envelope(bf_matrix, frames):
#     lateral_resolution = bf_matrix.shape[1]
#     samples = bf_matrix.shape[0]

#     env = np.zeros((samples, lateral_resolution, frames), dtype=bf_matrix.dtype)
#     for i in range(0, frames):
#         for j in range(0, lateral_resolution):
#             env[:,j,i] = envelope(bf_matrix[:,j,i], residual=None)
#     return env

In [ ]:
# if run_nsi_apodization:

#     bf_env_nsi = np.zeros((samples_for_acquisition_depth, samples_for_array_length, frames_to_process), dtype=bf_dtype)

#     memmap_bf_matrix_zm = np.load(f"{bf_data_path}{folder_name}/beamformed_apodized/{beamformed_apodized_matrix_name}_zm.npy")
#     memmap_bf_matrix_dc1 = np.load(f"{bf_data_path}{folder_name}/beamformed_apodized/{beamformed_apodized_matrix_name}_dc1.npy")
#     memmap_bf_matrix_dc2 = np.load(f"{bf_data_path}{folder_name}/beamformed_apodized/{beamformed_apodized_matrix_name}_dc2.npy")

#     memmap_bf_matrix_zm = get_envelope(memmap_bf_matrix_zm, frames=frames_to_process)
#     memmap_bf_matrix_dc1 = get_envelope(memmap_bf_matrix_dc1, frames=frames_to_process)
#     memmap_bf_matrix_dc2 = get_envelope(memmap_bf_matrix_dc2, frames=frames_to_process)

#     for i in range(0, frames_to_process):
#         bf_env_nsi[:,:,i] = (memmap_bf_matrix_dc1[:,:,i] + memmap_bf_matrix_dc2[:,:,i]) / 2 - memmap_bf_matrix_zm[:,:,i]

#     save_beamformed_data(bf_env_nsi, f"{beamformed_apodized_matrix_name}_env_nsi", destination=f"{bf_data_path}{folder_name}/beamformed_apodized/")
    
#     del memmap_bf_matrix_zm
#     del memmap_bf_matrix_dc1
#     del memmap_bf_matrix_dc2
#     del bf_env_nsi

# Sum frames

In [ ]:
# bf_frames_path = os.path.join(bf_data_path, folder_name, "beamformed_apodized") 
# bf_frames = np.load(f"{bf_frames_path}/{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}.npy")

# bf_final_path = os.path.join(bf_data_path, folder_name, "beamformed_final")
# bf_final = np.sum(bf_frames, axis=2)
# save_beamformed_data(bf_final, f"{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}", destination=f"{bf_final_path}/")

# del bf_final
# del bf_frames

## Show beamformed image

In [ ]:
# bf_image = np.load(f"{bf_data_path}{folder_name}/beamformed_final/{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}.npy")

# pixel_width = params["image"]["dx"]
# pixel_height = params["image"]["dz"]
# physical_aspect = pixel_width / pixel_height

# plt.figure(figsize=(10, 8))
# plt.imshow(np.abs(bf_image), cmap='viridis', aspect=physical_aspect)

# plt.title(f"Beamformed Image ({'env_nsi' if run_nsi_apodization else other_apodization})")

# plt.xlabel("Lateral samples")
# plt.ylabel("Depth (samples)")

# plt.colorbar(label="Amplitude") 

# if save_image:
#     plt.savefig(f"{bf_data_path}{folder_name}/images/{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}.png")
# plt.show()

## Observe RF

In [ ]:
rf_matrix_plot, debug_statement = load_rf_data(
            rf_matrix_path,
            matlab_rf_data_path,
            [0, 1],
            using_simulation_data,
            using_matlab_data, 
            len(params["rf"]["angles"])
        )

rf_matrix_plot = np.reshape(rf_matrix_plot, (rf_matrix_plot.shape[0], rf_matrix_plot.shape[1], rf_matrix_plot.shape[2]))

print(f"Matrix shape: {rf_matrix_plot.shape}")

#plotting
plt.imshow(rf_matrix_plot[:,:,0], cmap='viridis', aspect='auto')
plt.title('RF data summed')
plt.xlabel('Transducer idx')
plt.ylabel('Samples')
plt.colorbar(label='Amplitude')
plt.show()

NameError: name 'load_rf_data' is not defined